# C1. English LLM: distilgpt2 fine-tuned with LoRA on agricultural Q&A

**Model**: pretrained transformer (distilgpt2, 82M parameters) adapted with LoRA · **Report**: Section C · **Language**: English  
This is not the LSTM of Section B, Question 2: the two share no code, data or weights.


**Course**: ICS554 Natural Language Processing · Ashesi University
**Team**: MICS 2028 · Group 1
**Project**: Prosit 1 (Ankora AI Research Lab)
**Domain corpus**: `KisanVaani/agriculture-qa-english-only`, one row per distinct question
**Method**: LoRA ($r=8, \alpha=32$) on distilgpt2's attention projection `c_attn`, trained two ways

---
### What this notebook does
Training lives in `src/section_c_llm/train_lora.py` (about 15 minutes on a laptop CPU). It trains a *standard* adapter (loss on every token) and a *prompt-masked* adapter (loss on answer tokens only) on the same 500 training questions, then scores base, standard and masked on the same 222 held-out questions. This notebook loads the saved adapters, re-scores them, and shows seeded completions.

1. **Data**: deduplicate by question before splitting (the raw corpus repeats each question about ten times).
2. **LoRA configuration**: what is trained and how many parameters that is.
3. **Results**: three perplexities per model, re-computed here from the saved adapters.
4. **Completions**: the same seeded prompts for every model.
5. **Decoding strategies** and **prompt-loss masking**: what each changes, and what it does not.

In [ ]:
import sys
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch
from peft import PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, set_seed

from src.section_c_llm.prepare_data import load_split, PROCESSED_DIR
from src.section_c_llm.train_lora import (
    BASE_MODEL, MODEL_DIRS, PROMPTS, encode, lora_config, perplexity, prompt_len, sample, tokenizer,
)

RANDOM_SEED = 42
set_seed(RANDOM_SEED)
print("PyTorch", torch.__version__)

## 1. Data: one row per question
`src/section_c_llm/prepare_data.py` keeps the first row of every distinct question *before* shuffling, so no test question can also be in training. The splits are JSONL because some answers contain blank lines, which used to break a single Q&A pair into several fragments.

In [ ]:
print(json.loads((PROCESSED_DIR / "stats.json").read_text()))
train_texts, val_texts, test_texts = load_split("train", 500), load_split("val"), load_split("test")
print(f"Used for training: {len(train_texts)} | val: {len(val_texts)} | test: {len(test_texts)}")
print("\nSample test record:\n" + test_texts[0])

## 2. LoRA configuration
LoRA freezes the pretrained weight $W_0$ and learns a low-rank update $\Delta W = \frac{\alpha}{r} BA$. In GPT-2, `c_attn` is one fused projection that produces the query, key and value vectors, so the adapter touches all three.

In [ ]:
demo = get_peft_model(AutoModelForCausalLM.from_pretrained(BASE_MODEL), lora_config())
demo.print_trainable_parameters()

## 3. Results
The numbers below come from `results/section_c_llm/lora_results.json`, written by the training script. The next cell re-scores the saved adapters and should print the same values.

- **full_ppl**: every token of the held-out Q&A text
- **answer_ppl**: answer tokens only, given the question (the part that is domain knowledge rather than the Question/Answer template)
- **wikitext_ppl**: 200 general-English paragraphs; a rise here is what the adaptation costs

In [ ]:
report = json.loads((REPO_ROOT / "results" / "section_c_llm" / "lora_results.json").read_text())
metrics = ["full_ppl", "answer_ppl", "wikitext_ppl"]
pd.DataFrame({name: {m: r[m] for m in metrics} for name, r in report["results"].items()}).T

In [ ]:
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
models = {
    "base": base,
    "standard": PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE_MODEL), str(MODEL_DIRS["standard"])),
    "masked": PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE_MODEL), str(MODEL_DIRS["masked"])),
}
rescored = {name: {"full_ppl": perplexity(m, test_texts), "answer_ppl": perplexity(m, test_texts, answer_only=True)}
            for name, m in models.items()}
pd.DataFrame(rescored).T

## 4. Seeded completions
Same prompt, same seed, same sampling settings (temperature 0.7, top-p 0.9) for every model. Read them for fluency *and* correctness: an 82M-parameter model trained on 500 examples picks up the answer style well before it gets the facts right.

In [ ]:
for prompt in PROMPTS:
    print("=" * 80 + "\n" + prompt)
    for name, m in models.items():
        print(f"\n[{name}] {sample(m, prompt)}")

## 5. Decoding strategies
Sampling from a small model can loop, repeating one sentence until the token limit. A repetition penalty or `no_repeat_ngram_size=3` stops the loop. Distinct-3 (unique word trigrams / all trigrams) measures that, and under 3-gram blocking it is close to 1 by construction. It does not measure whether the advice is right. The full benchmark (4 prompts, 5 seeds each) is `src/section_c_llm/benchmark_decoding.py`.

In [ ]:
decoding_configs = {
    "Unpenalized (T=0.7)": {"do_sample": True, "temperature": 0.7, "top_p": 0.9},
    "Repetition penalty 1.3": {"do_sample": True, "temperature": 0.7, "top_p": 0.9, "repetition_penalty": 1.3},
    "3-gram blocking": {"do_sample": True, "temperature": 0.7, "top_p": 0.9, "no_repeat_ngram_size": 3},
    "Greedy + penalty + block": {"do_sample": False, "repetition_penalty": 1.25, "no_repeat_ngram_size": 3},
}
prompt = PROMPTS[2]
ids = tokenizer(prompt, return_tensors="pt").input_ids
print(prompt)
for name, kwargs in decoding_configs.items():
    set_seed(RANDOM_SEED)
    with torch.no_grad():
        out = models["standard"].generate(ids, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id, **kwargs)
    print(f"\n[{name}] {tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()}")

## 6. Prompt-loss masking
With masking, the question tokens get label `-100`, so `CrossEntropyLoss` ignores them and training only optimises $P(\text{answer} \mid \text{question})$. Compare the *masked* and *standard* rows in section 3: masking helps only if its **answer_ppl** is lower. It also stops the model learning to predict questions, which matters if the adapted LM is meant to score whole transcripts (for example, farmers' spoken questions in an ASR system).

In [ ]:
row = encode([train_texts[0]], mask_prompt=True)[0]["labels"]
k = prompt_len(train_texts[0])
print(f"Prompt tokens masked: {k} | answer tokens trained: {sum(t != -100 for t in row[k:])}")
print("Masked part:", repr(tokenizer.decode(tokenizer(train_texts[0]).input_ids[:k])))